# Calibrated Prediction Intervals for Nuclear Masses under Extrapolation

**Goal.** Build prediction intervals for nuclear binding energies whose coverage holds on nuclei measured *after* the training data, i.e. under the covariate shift that real extrapolation always involves.

**Pipeline**

1. Download AME2012, AME2016 and AME2020 mass tables.
2. Temporal split: **train** = measured in AME2012, **calibration** = new in AME2016, **test** = new in AME2020. Targets always come from AME2020 (most precise values).
3. Physics base model: a liquid-drop formula fitted on the training set only.
4. Residual learner: gradient-boosted trees (point + quantile).
5. Interval methods compared:
   - **GP**: Gaussian-process credible interval (Bayesian baseline).
   - **CQR-random**: conformalized quantile regression calibrated on a *random* part of the training set (the usual exchangeable recipe).
   - **CQR-temporal**: calibrated on the AME2016 additions.
   - **CQR-temporal-scaled (proposed)**: temporal calibration of *distance-normalised* scores, where the growth of error with extrapolation distance is learned by **frontier peeling** (holding out the outer layers of the known chart), so bands widen farther from known nuclei.
6. Coverage and width as functions of extrapolation distance, then an exported table of intervals for nuclei not yet measured.

Runs on CPU in a few minutes. On Kaggle, turn **Internet ON** in the notebook settings.

In [ ]:
import os, io, re, time, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)
ALPHA = 0.10                      # target miscoverage -> 90% intervals
DATA_DIR = "ame_data"
os.makedirs(DATA_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 1. Download the AME tables

The cell tries, in order: (1) a copy already in `ame_data/`, (2) any matching file under `/kaggle/input` (an uploaded Kaggle dataset), (3) several mirrors with browser-like headers. Some servers return **403 Forbidden** to scripts; if every mirror fails, download the three files in your browser from the AMDC pages below and either put them in `ame_data/` or upload them as a Kaggle dataset, then rerun.

* AME2012: `mass.mas12` — https://www-nds.iaea.org/amdc/ (section AME2012) or http://amdc.in2p3.fr/masstables/Ame2012/mass.mas12
* AME2016: `mass16.txt` — https://www-nds.iaea.org/amdc/ (section AME2016)
* AME2020: `mass_1.mas20.txt` (or `mass_1.mas20`) — https://www-nds.iaea.org/amdc/ (section AME2020)

In [ ]:
import glob, shutil

AME_URLS = {
    2012: ["https://www-nds.iaea.org/amdc/ame2012/mass.mas12",
           "http://amdc.in2p3.fr/masstables/Ame2012/mass.mas12",
           "https://amdc.impcas.ac.cn/masstables/Ame2012/mass.mas12"],
    2016: ["https://www-nds.iaea.org/amdc/ame2016/mass16.txt",
           "http://amdc.in2p3.fr/masstables/Ame2016/mass16.txt",
           "https://amdc.impcas.ac.cn/masstables/Ame2016/mass16.txt"],
    2020: ["https://www-nds.iaea.org/amdc/ame2020/mass_1.mas20.txt",
           "https://www-nds.iaea.org/amdc/ame2020/mass_1.mas20",
           "https://amdc.impcas.ac.cn/masstables/Ame2020/mass_1.mas20.txt"],
}
AME_FILES = {2012: "mass.mas12", 2016: "mass16.txt", 2020: "mass_1.mas20.txt"}
ALT_NAMES = {2012: ["mass.mas12", "mass.mas12.txt", "mass12.txt"],
             2016: ["mass16.txt", "mass.mas16", "mass16"],
             2020: ["mass_1.mas20.txt", "mass_1.mas20", "mass.mas20"]}
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "text/plain,text/html;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www-nds.iaea.org/amdc/",
}


def looks_like_ame(data):
    txt = data[:200_000].decode("latin-1", errors="ignore")
    return len(data) > 100_000 and ("MASS" in txt.upper() or "1n" in txt)


def fetch(url):
    try:
        import requests
        r = requests.get(url, headers=HEADERS, timeout=60)
        r.raise_for_status()
        return r.content
    except ImportError:
        req = urllib.request.Request(url, headers=HEADERS)
        with urllib.request.urlopen(req, timeout=60) as r:
            return r.read()


def download(year):
    path = os.path.join(DATA_DIR, AME_FILES[year])
    if os.path.exists(path) and os.path.getsize(path) > 100_000:
        print(f"AME{year}: already in {path}")
        return path
    # uploaded Kaggle dataset or any local copy
    for root in ["/kaggle/input", ".", os.path.expanduser("~")]:
        for name in ALT_NAMES[year]:
            hits = glob.glob(os.path.join(root, "**", name), recursive=True)
            hits = [h for h in hits if os.path.getsize(h) > 100_000 and os.path.abspath(h) != os.path.abspath(path)]
            if hits:
                shutil.copy(hits[0], path)
                print(f"AME{year}: copied from {hits[0]}")
                return path
    for url in AME_URLS[year]:
        try:
            data = fetch(url)
            if not looks_like_ame(data):
                raise ValueError(f"not an AME table ({len(data)} bytes)")
            with open(path, "wb") as f:
                f.write(data)
            print(f"AME{year}: downloaded {len(data)/1e6:.2f} MB from {url}")
            return path
        except Exception as e:
            print(f"AME{year}: failed from {url} -> {e}")
    return None


paths = {y: download(y) for y in (2012, 2016, 2020)}
missing = [y for y, p in paths.items() if p is None]
if missing:
    raise FileNotFoundError(
        "Could not get AME " + ", ".join(map(str, missing)) +
        ". Download them in your browser (links in the cell above) and place them in "
        f"'{DATA_DIR}/' as {[AME_FILES[y] for y in missing]}, or upload them as a Kaggle dataset, then rerun.")

## 2. Parse the tables

All three editions share the same leading columns (`N`, `Z`, `A` in characters 5–19, mass excess starting at character 29). A `#` after a value marks an **estimate from systematics**, not a measurement; those are kept separately.

Binding energy (keV) from the atomic mass excess Δ:

$$B(Z,N) = Z\,\Delta_{^1\mathrm{H}} + N\,\Delta_n - \Delta(Z,N),\qquad \Delta_{^1\mathrm{H}} = 7288.971064,\ \Delta_n = 8071.318062\ \text{keV}.$$

In [ ]:
DELTA_H, DELTA_N = 7288.971064, 8071.318062   # keV, AME2020


def parse_ame(path):
    rows = []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            if len(line) < 40:
                continue
            try:
                N = int(line[4:9]); Z = int(line[9:14]); A = int(line[14:19])
            except ValueError:
                continue
            if N + Z != A or A < 1:
                continue
            toks = line[28:].split()
            if not toks:
                continue
            tok = toks[0]
            estimated = "#" in tok
            try:
                me = float(tok.replace("#", ""))
            except ValueError:
                continue
            rows.append((Z, N, A, me, estimated))
    df = pd.DataFrame(rows, columns=["Z", "N", "A", "mass_excess", "estimated"])
    df = df.drop_duplicates(["Z", "N"]).reset_index(drop=True)
    df["B"] = df.Z * DELTA_H + df.N * DELTA_N - df.mass_excess      # keV
    return df


ame = {y: parse_ame(p) for y, p in paths.items()}
for y, d in ame.items():
    print(f"AME{y}: {len(d):5d} nuclides, {(~d.estimated).sum():5d} measured, {d.estimated.sum():4d} estimated (#)")

In [ ]:
# Sanity check: B/A of 56Fe and 208Pb should be ~8790 and ~7867 keV
d20 = ame[2020]
for z, a in [(26, 56), (82, 208)]:
    r = d20[(d20.Z == z) & (d20.A == a)]
    if len(r):
        print(f"Z={z} A={a}: B/A = {r.B.iloc[0] / a:.3f} keV")
    else:
        print(f"Z={z} A={a}: not found -> check the parser")

## 3. Temporal split

Keep nuclei with Z, N ≥ 8 (standard in the literature). A nucleus is placed in the first edition in which it was **measured**:

| Set | Definition | Role |
|---|---|---|
| train | measured in AME2012 | fit base model and learners |
| calib | measured in AME2016 but not AME2012 | conformal calibration |
| test  | measured in AME2020 but not AME2016 | final evaluation |
| frontier | only estimated (#) in AME2020 | export predicted intervals |

Targets always use AME2020 values.

In [ ]:
def measured_set(d):
    m = d[(~d.estimated) & (d.Z >= 8) & (d.N >= 8)]
    return set(zip(m.Z, m.N))

S12, S16, S20 = measured_set(ame[2012]), measured_set(ame[2016]), measured_set(ame[2020])

data = d20[(d20.Z >= 8) & (d20.N >= 8)].copy()
key = list(zip(data.Z, data.N))
data["split"] = np.select(
    [[k in S12 for k in key], [k in S16 for k in key], [k in S20 for k in key]],
    ["train", "calib", "test"], default="frontier")
# a nucleus measured in 2012 but estimated in 2020 (rare) is dropped
data = data[~((data.split != "frontier") & data.estimated)].reset_index(drop=True)
print(data.split.value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = {"train": "#9aa5b1", "calib": "#2f6fb0", "test": "#d9480f", "frontier": "#e9ecef"}
for s in ["frontier", "train", "calib", "test"]:
    q = data[data.split == s]
    ax.scatter(q.N, q.Z, s=4 if s in ("train", "frontier") else 9, c=colors[s], label=f"{s} ({len(q)})", marker="s")
ax.set_xlabel("N"); ax.set_ylabel("Z"); ax.set_title("Temporal split on the nuclear chart")
ax.legend(markerscale=3, loc="upper left"); plt.tight_layout(); plt.show()

## 4. Physics base model (liquid drop) fitted on the training set only

$$B_{\mathrm{LDM}} = a_v A - a_s A^{2/3} - a_c \frac{Z(Z-1)}{A^{1/3}} - a_a \frac{(N-Z)^2}{A} + a_{ss}\frac{(N-Z)^2}{A^{4/3}} + a_p\,\delta\,A^{-1/2}$$

with δ = +1 (even–even), −1 (odd–odd), 0 otherwise. The ML model then learns the residual $y = B_{\mathrm{exp}} - B_{\mathrm{LDM}}$.

> To use a stronger base model (WS4, FRDM2012, HFB-24), load its table into a DataFrame with columns `Z, N, B_theory` (keV) and set `BASE_TABLE` below. Nuclei missing from that table are dropped.

In [ ]:
BASE_TABLE = None   # e.g. pd.read_csv("ws4.csv")  with columns Z, N, B_theory (keV)


def ldm_design(Z, N):
    Z = np.asarray(Z, float); N = np.asarray(N, float); A = Z + N
    delta = np.where((Z % 2 == 0) & (N % 2 == 0), 1.0, np.where((Z % 2 == 1) & (N % 2 == 1), -1.0, 0.0))
    return np.column_stack([A, -A ** (2 / 3), -Z * (Z - 1) / A ** (1 / 3), -(N - Z) ** 2 / A,
                            (N - Z) ** 2 / A ** (4 / 3), delta * A ** -0.5])


tr = data.split == "train"
if BASE_TABLE is None:
    coef, *_ = np.linalg.lstsq(ldm_design(data.Z[tr], data.N[tr]), data.B[tr], rcond=None)
    data["B_theory"] = ldm_design(data.Z, data.N) @ coef
    print("LDM coefficients (MeV):", np.round(coef / 1000, 3))
else:
    data = data.merge(BASE_TABLE[["Z", "N", "B_theory"]], on=["Z", "N"], how="inner")

data["y"] = (data.B - data.B_theory) / 1000.0            # residual in MeV
for s in ["train", "calib", "test"]:
    print(f"base-model rms on {s:6s}: {np.sqrt(np.mean(data.y[data.split == s] ** 2)):.3f} MeV")

## 5. Features and extrapolation distance

Features: Z, N, A, isospin asymmetry, parities, distance to nearest magic numbers, and the Casten valence product $P = \nu_p\nu_n/(\nu_p+\nu_n)$.

**Extrapolation distance** $d$ = Manhattan distance $|\Delta Z| + |\Delta N|$ to the nearest *training* nucleus. It is the covariate-shift score used by the scaled calibration.

In [ ]:
MAGIC = np.array([2, 8, 20, 28, 50, 82, 126, 184])


def valence(x):
    x = np.asarray(x)[:, None]
    lower = np.where(MAGIC[None, :] <= x, MAGIC[None, :], -1).max(1)
    upper = np.where(MAGIC[None, :] > x, MAGIC[None, :], 10 ** 6).min(1)
    xv = x[:, 0]
    return np.minimum(xv - lower, upper - xv), xv - lower, upper - lower


def features(df):
    Z, N = df.Z.values.astype(float), df.N.values.astype(float); A = Z + N
    vz, pz, wz = valence(df.Z.values); vn, pn, wn = valence(df.N.values)
    P = np.where(vz + vn > 0, vz * vn / np.maximum(vz + vn, 1), 0.0)
    return np.column_stack([Z, N, A, (N - Z) / A, Z % 2, N % 2, vz, vn, pz / wz, pn / wn, P])


FEAT_NAMES = ["Z", "N", "A", "I", "Zodd", "Nodd", "vZ", "vN", "fillZ", "fillN", "P"]


def manhattan_to(ref_zn, zn):
    ref = np.asarray(ref_zn); out = np.empty(len(zn), int)
    for i, (z, n) in enumerate(zn):
        out[i] = np.min(np.abs(ref[:, 0] - z) + np.abs(ref[:, 1] - n))
    return out


X_all = features(data)
train_zn = data.loc[tr, ["Z", "N"]].values
data["dist"] = manhattan_to(train_zn, data[["Z", "N"]].values)
print(data.groupby("split").dist.describe()[["count", "mean", "50%", "max"]])

## 6. Residual learners

* Point model: gradient boosting (squared loss).
* Quantile models: gradient boosting with pinball loss at α/2 and 1−α/2.
* GP baseline on (Z, N, parities) with RBF + white-noise kernel.

**Plug-in for your own quantile model (e.g. NDT).** Any object with `fit(X, y)` and `predict_quantiles(X) -> (lo, hi)` can replace `QuantileGB` below.

In [ ]:
masks = {s: (data.split == s).values for s in ["train", "calib", "test", "frontier"]}
X = {s: X_all[m] for s, m in masks.items()}
y = {s: data.y.values[m] for s, m in masks.items()}
dist = {s: data.dist.values[m] for s, m in masks.items()}

GB_KW = dict(max_iter=600, learning_rate=0.04, max_leaf_nodes=15, min_samples_leaf=8,
             l2_regularization=1.0, random_state=SEED)


class QuantileGB:
    def __init__(self, alpha=ALPHA):
        self.lo = HistGradientBoostingRegressor(loss="quantile", quantile=alpha / 2, **GB_KW)
        self.hi = HistGradientBoostingRegressor(loss="quantile", quantile=1 - alpha / 2, **GB_KW)

    def fit(self, X, y):
        self.lo.fit(X, y); self.hi.fit(X, y); return self

    def predict_quantiles(self, X):
        lo, hi = self.lo.predict(X), self.hi.predict(X)
        return np.minimum(lo, hi), np.maximum(lo, hi)


t0 = time.time()
point = HistGradientBoostingRegressor(loss="squared_error", **GB_KW).fit(X["train"], y["train"])
qgb = QuantileGB().fit(X["train"], y["train"])
print(f"boosting fitted in {time.time() - t0:.1f}s")

rms = lambda a, b: np.sqrt(np.mean((a - b) ** 2))
for s in ["train", "calib", "test"]:
    print(f"{s:6s}: base rms {rms(y[s], 0):.3f} MeV -> refined rms {rms(y[s], point.predict(X[s])):.3f} MeV")

In [ ]:
# Gaussian process baseline (inputs: scaled Z, N + parities)
gp_cols = [0, 1, 4, 5]
mu_g, sd_g = X["train"][:, gp_cols].mean(0), X["train"][:, gp_cols].std(0)
gp_in = lambda Xs: (Xs[:, gp_cols] - mu_g) / sd_g

kernel = C(1.0) * RBF(length_scale=[0.2, 0.2, 1.0, 1.0]) + WhiteKernel(0.1)
t0 = time.time()
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=1,
                              random_state=SEED).fit(gp_in(X["train"]), y["train"])
print(f"GP fitted in {time.time() - t0:.1f}s; kernel: {gp.kernel_}")
z = 1.6448536269514722   # 90% normal quantile
for s in ["calib", "test"]:
    m, sd = gp.predict(gp_in(X[s]), return_std=True)
    print(f"{s:6s}: GP rms {rms(y[s], m):.3f} MeV, 90% coverage {np.mean(np.abs(y[s] - m) <= z * sd):.3f}")

## 7. Conformal calibration

CQR score: $s_i = \max\{\hat q_{lo}(x_i) - y_i,\; y_i - \hat q_{hi}(x_i)\}$; interval $[\hat q_{lo} - \hat c,\ \hat q_{hi} + \hat c]$ with $\hat c$ the $\lceil (n+1)(1-\alpha)\rceil/n$ empirical quantile of the calibration scores.

* **CQR-random:** quantile models refit on 80% of the training set, $\hat c$ from the other 20% (the usual exchangeable recipe).
* **CQR-temporal:** $\hat c$ from the AME2016 additions.
* **CQR-temporal-scaled (proposed):** scores are normalised by a growth function $g(d) = 1 + \gamma (d - 1)$, so the interval is $[\hat q_{lo} - \hat c\,g(d),\ \hat q_{hi} + \hat c\,g(d)]$.

**How γ is learned — frontier peeling.** The AME2016 additions sit almost all at $d = 1$, so they cannot reveal how errors grow farther out. We therefore imitate extrapolation *inside* the training set: remove the outer $K$ layers of the training chart, fit the quantile model on the remaining core, and measure the 90% score quantile on the peeled nuclei at distances $d = 1, \dots, K$ from the core. γ is the least-squares slope of $q_d / q_1$ against $d - 1$ (clipped at 0). The constant $\hat c$ is then calibrated on the real temporal set.

In [ ]:
def conformal_q(scores, alpha=ALPHA):
    n = len(scores)
    k = min(int(np.ceil((n + 1) * (1 - alpha))), n)
    return np.sort(scores)[k - 1]


def cqr_scores(model, Xc, yc):
    lo, hi = model.predict_quantiles(Xc)
    return np.maximum(lo - yc, yc - hi)


# --- CQR-random -------------------------------------------------------------
idx_fit, idx_cal = train_test_split(np.arange(len(y["train"])), test_size=0.2, random_state=SEED)
q_rand = QuantileGB().fit(X["train"][idx_fit], y["train"][idx_fit])
c_rand = conformal_q(cqr_scores(q_rand, X["train"][idx_cal], y["train"][idx_cal]))

# --- CQR-temporal -----------------------------------------------------------
s_cal = cqr_scores(qgb, X["calib"], y["calib"])
c_temp = conformal_q(s_cal)
print(f"c_random = {c_rand:.3f} MeV, c_temporal = {c_temp:.3f} MeV")

In [ ]:
# --- Frontier peeling: learn how CQR scores grow with extrapolation distance ---
K_PEEL = 3


def depth_inside(zn_set):
    # depth of each nucleus = Manhattan distance to the nearest (Z,N>=8) position NOT in the set
    zn = np.array(sorted(zn_set))
    zmin, nmin = 8, 8
    zmax, nmax = zn[:, 0].max() + K_PEEL + 2, zn[:, 1].max() + K_PEEL + 2
    grid = np.zeros((zmax + 1, nmax + 1), bool)
    grid[zn[:, 0], zn[:, 1]] = True
    holes = np.argwhere(~grid)
    holes = holes[(holes[:, 0] >= zmin - 1) & (holes[:, 1] >= nmin - 1)]
    return zn, manhattan_to(holes, zn)


def learn_gamma(Xtr, ytr, zn_tr, k=K_PEEL, alpha=ALPHA, verbose=True):
    zn_sorted, depth = depth_inside(set(map(tuple, zn_tr)))
    depth_map = {tuple(p): d for p, d in zip(zn_sorted, depth)}
    dep = np.array([depth_map[tuple(p)] for p in zn_tr])
    core = dep > k
    qm = QuantileGB(alpha).fit(Xtr[core], ytr[core])
    d_core = manhattan_to(zn_tr[core], zn_tr[~core])
    s = cqr_scores(qm, Xtr[~core], ytr[~core])
    ds, qs = [], []
    for d in range(1, k + 1):
        m = d_core == d
        if m.sum() >= 20:
            ds.append(d); qs.append(np.quantile(s[m], 1 - alpha))
    ds, qs = np.array(ds), np.array(qs)
    if len(ds) < 2 or qs[0] <= 0:
        return 0.0, ds, qs
    ratio = qs / qs[0]
    gamma = max(0.0, float(np.sum((ds - 1) * (ratio - 1)) / max(np.sum((ds - 1) ** 2), 1e-9)))
    if verbose:
        print(f"core: {core.sum()} nuclei, peeled: {(~core).sum()}")
        for d, q in zip(ds, qs):
            print(f"  d={d}: n={np.sum(d_core == d):4d}, 90% score quantile = {q:.3f} MeV")
    return gamma, ds, qs


zn_train = data.loc[tr, ["Z", "N"]].values
gamma, peel_d, peel_q = learn_gamma(X["train"], y["train"], zn_train)
g = lambda d: 1.0 + gamma * (np.maximum(d, 1) - 1)
c_scaled = conformal_q(s_cal / g(dist["calib"]))
print(f"gamma = {gamma:.3f}  ->  g(d) = 1 + {gamma:.3f}(d-1);  c_scaled = {c_scaled:.3f} MeV")

In [ ]:
def intervals(s):
    Xs, ds = X[s], dist[s]
    out = {}
    m, sd = gp.predict(gp_in(Xs), return_std=True)
    out["GP"] = (m - z * sd, m + z * sd)
    lo, hi = q_rand.predict_quantiles(Xs);  out["CQR-random"] = (lo - c_rand, hi + c_rand)
    lo, hi = qgb.predict_quantiles(Xs);     out["CQR-temporal"] = (lo - c_temp, hi + c_temp)
    out["CQR-temporal-scaled"] = (lo - c_scaled * g(ds), hi + c_scaled * g(ds))
    return out


def winkler(y_, lo, hi, alpha=ALPHA):
    w = hi - lo
    return np.mean(w + (2 / alpha) * ((lo - y_) * (y_ < lo) + (y_ - hi) * (y_ > hi)))


rows = []
for s in ["calib", "test"]:
    for name, (lo, hi) in intervals(s).items():
        cov = (y[s] >= lo) & (y[s] <= hi)
        rows.append(dict(set=s, method=name, coverage=cov.mean(), mean_width_MeV=np.mean(hi - lo),
                         winkler=winkler(y[s], lo, hi)))
res = pd.DataFrame(rows)
print(f"Target coverage = {1 - ALPHA:.2f}")
res.round(3)

## 8. Coverage and width versus extrapolation distance (test set)

The key plot of the paper: a method is useful only if its coverage stays near 90% at **every** distance, not just on average.

In [ ]:
def dist_label(d):
    return np.where(d >= 4, "4+", d.astype(str))


ints = intervals("test")
lab = dist_label(dist["test"])
order = [b for b in ["1", "2", "3", "4+"] if b in set(lab)]
rows = []
for name, (lo, hi) in ints.items():
    for b in order:
        m = lab == b
        rows.append(dict(method=name, d=b, n=m.sum(),
                         coverage=np.mean((y["test"][m] >= lo[m]) & (y["test"][m] <= hi[m])),
                         width=np.mean(hi[m] - lo[m])))
by_d = pd.DataFrame(rows)
print("test nuclei per distance:", dict(zip(*np.unique(lab, return_counts=True))))
display(by_d.pivot(index="d", columns="method", values="coverage").round(3))
display(by_d.pivot(index="d", columns="method", values="width").round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name in ints:
    q = by_d[by_d.method == name].set_index("d").reindex(order)
    axes[0].plot(order, q.coverage, marker="o", label=name)
    axes[1].plot(order, q.width, marker="o", label=name)
axes[0].axhline(1 - ALPHA, color="k", ls="--", lw=1)
axes[0].set_ylim(0, 1.02); axes[0].set_ylabel("empirical coverage"); axes[0].set_xlabel("extrapolation distance d")
axes[1].set_ylabel("mean 90% interval width (MeV)"); axes[1].set_xlabel("extrapolation distance d")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# Point accuracy versus distance
pt = point.predict(X["test"])
tab = pd.DataFrame({"d": lab, "base": y["test"] ** 2, "refined": (y["test"] - pt) ** 2})
print(tab.groupby("d")[["base", "refined"]].mean().pow(0.5).round(3).rename(columns=lambda c: c + "_rms_MeV"))

## 9. Robustness: bootstrap the training set

Gradient boosting here is deterministic, so instead of seeds we refit everything on bootstrap resamples of the training nuclei and report mean ± sd of test coverage and width.

In [ ]:
N_BOOT = 5
rep = []
for r in range(N_BOOT):
    rb = np.random.default_rng(100 + r)
    ib = rb.integers(0, len(y["train"]), len(y["train"]))
    qb = QuantileGB().fit(X["train"][ib], y["train"][ib])
    sc = cqr_scores(qb, X["calib"], y["calib"])
    ct = conformal_q(sc)
    cs = conformal_q(sc / g(dist["calib"]))
    lo_t, hi_t = qb.predict_quantiles(X["test"])
    gt = g(dist["test"])
    rep.append(dict(boot=r,
                    cov_temporal=np.mean((y["test"] >= lo_t - ct) & (y["test"] <= hi_t + ct)),
                    cov_scaled=np.mean((y["test"] >= lo_t - cs * gt) & (y["test"] <= hi_t + cs * gt)),
                    width_temporal=np.mean(hi_t - lo_t + 2 * ct),
                    width_scaled=np.mean(hi_t - lo_t + 2 * cs * gt)))
rep = pd.DataFrame(rep)
print(rep.round(3)); print("\n", rep.drop(columns="boot").agg(["mean", "std"]).round(3))

## 10. Export intervals for unmeasured nuclei (frontier)

Final model: learners refit on train + calib, γ re-learned by peeling on that larger set, $\hat c$ kept from the temporal calibration, and distances measured from all nuclei measured in AME2020. Output: `predicted_masses_intervals.csv` with binding energy and 90% interval (MeV) for every AME2020 `#` nucleus — the seed of an ADNDT-style table.

Rows flagged `extrapolated_beyond_calibration` lie farther than any distance used to learn γ; their intervals rely on extending the linear growth and should be reported as such.

In [ ]:
known = masks["train"] | masks["calib"]
q_final = QuantileGB().fit(X_all[known], data.y.values[known])
p_final = HistGradientBoostingRegressor(loss="squared_error", **GB_KW).fit(X_all[known], data.y.values[known])
gamma_f, _, _ = learn_gamma(X_all[known], data.y.values[known], data.loc[known, ["Z", "N"]].values, verbose=False)
g_f = lambda d: 1.0 + gamma_f * (np.maximum(d, 1) - 1)
print(f"final gamma = {gamma_f:.3f}")

fr = data[masks["frontier"]].copy()
measured_zn = data.loc[masks["train"] | masks["calib"] | masks["test"], ["Z", "N"]].values
fr["dist"] = manhattan_to(measured_zn, fr[["Z", "N"]].values)
lo, hi = q_final.predict_quantiles(X_all[masks["frontier"]])
half = c_scaled * g_f(fr.dist.values)
base = fr.B_theory.values / 1000
fr["B_pred_MeV"] = base + p_final.predict(X_all[masks["frontier"]])
fr["B_lo90_MeV"] = base + lo - half
fr["B_hi90_MeV"] = base + hi + half
fr["B_AME2020_systematics_MeV"] = fr.B / 1000
fr["extrapolated_beyond_calibration"] = fr.dist > K_PEEL
out = fr[["Z", "N", "A", "dist", "B_pred_MeV", "B_lo90_MeV", "B_hi90_MeV",
          "B_AME2020_systematics_MeV", "extrapolated_beyond_calibration"]].sort_values(["Z", "N"])
out.to_csv("predicted_masses_intervals.csv", index=False)
inside = np.mean((out.B_AME2020_systematics_MeV >= out.B_lo90_MeV) & (out.B_AME2020_systematics_MeV <= out.B_hi90_MeV))
print(f"{len(out)} frontier nuclei exported; {inside:.1%} of AME2020 systematic estimates fall inside our 90% intervals")
print(f"{out.extrapolated_beyond_calibration.sum()} nuclei lie beyond d={K_PEEL} (interpret with care)")
out.head()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(data.N[known], data.Z[known], s=3, c="#ced4da", marker="s", label="measured")
sc = ax.scatter(out.N, out.Z, s=6, c=out.B_hi90_MeV - out.B_lo90_MeV, cmap="viridis", marker="s")
plt.colorbar(sc, label="90% interval width (MeV)")
ax.set_xlabel("N"); ax.set_ylabel("Z"); ax.set_title("Interval width on unmeasured nuclei"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

## 11. Save all results to Excel

One workbook, `nuclear_mass_results.xlsx`, with a sheet per result table. (Modern Excel format `.xlsx`; it opens in Excel, LibreOffice and Google Sheets.)

In [ ]:
try:
    import openpyxl  # noqa: F401
except ImportError:
    !pip install -q openpyxl

XLSX_PATH = "nuclear_mass_results.xlsx"

config = pd.DataFrame({
    "parameter": ["alpha (miscoverage)", "target coverage", "base model", "learner", "K_PEEL",
                  "gamma (temporal run)", "gamma (final run)", "c_random (MeV)", "c_temporal (MeV)",
                  "c_scaled (MeV)", "n_train", "n_calib", "n_test", "n_frontier"],
    "value": [ALPHA, 1 - ALPHA, "LDM" if BASE_TABLE is None else "user table", "HistGradientBoosting",
              K_PEEL, round(gamma, 4), round(gamma_f, 4), round(c_rand, 4), round(c_temp, 4),
              round(c_scaled, 4), masks["train"].sum(), masks["calib"].sum(), masks["test"].sum(),
              masks["frontier"].sum()],
})
cov_table = by_d.pivot(index="d", columns="method", values="coverage").reset_index()
width_table = by_d.pivot(index="d", columns="method", values="width").reset_index()
rms_table = tab.groupby("d")[["base", "refined"]].mean().pow(0.5).reset_index() \
    .rename(columns={"base": "base_rms_MeV", "refined": "refined_rms_MeV"})
peel_table = pd.DataFrame({"d": peel_d, "score_q90_MeV": peel_q,
                           "ratio_to_d1": peel_q / peel_q[0] if len(peel_q) else peel_q,
                           "g(d)_fitted": g(peel_d)})
boot_summary = rep.drop(columns="boot").agg(["mean", "std"]).reset_index().rename(columns={"index": "stat"})

# per-nucleus test predictions (useful for plots in the paper)
ints_test = intervals("test")
test_df = data.loc[masks["test"], ["Z", "N", "A", "dist"]].copy()
test_df["y_residual_MeV"] = y["test"]
test_df["point_pred_MeV"] = point.predict(X["test"])
for name, (lo_, hi_) in ints_test.items():
    test_df[f"{name}_lo"] = lo_
    test_df[f"{name}_hi"] = hi_
    test_df[f"{name}_covered"] = (y["test"] >= lo_) & (y["test"] <= hi_)

with pd.ExcelWriter(XLSX_PATH, engine="openpyxl") as xw:
    config.to_excel(xw, sheet_name="config", index=False)
    res.round(4).to_excel(xw, sheet_name="summary", index=False)
    cov_table.round(4).to_excel(xw, sheet_name="coverage_by_distance", index=False)
    width_table.round(4).to_excel(xw, sheet_name="width_by_distance", index=False)
    rms_table.round(4).to_excel(xw, sheet_name="point_rms_by_distance", index=False)
    peel_table.round(4).to_excel(xw, sheet_name="frontier_peeling", index=False)
    rep.round(4).to_excel(xw, sheet_name="bootstrap_runs", index=False)
    boot_summary.round(4).to_excel(xw, sheet_name="bootstrap_summary", index=False)
    test_df.round(4).to_excel(xw, sheet_name="test_predictions", index=False)
    out.round(4).to_excel(xw, sheet_name="frontier_predictions", index=False)
    # widen columns for readability
    for ws in xw.book.worksheets:
        for col in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0 for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(10, width + 2), 40)
        ws.freeze_panes = "A2"

print(f"saved {XLSX_PATH} with sheets:", pd.ExcelFile(XLSX_PATH).sheet_names)

## 12. Next steps

1. Replace the liquid-drop base with WS4 / FRDM2012 / HFB-24 (`BASE_TABLE`) and compare.
2. Swap `QuantileGB` for your NDT quantile network.
3. Add **weighted** conformal prediction (likelihood-ratio weights from a classifier separating calib vs test features) as a second shift-aware method.
4. Evaluate derived quantities: $S_n$, $S_{2n}$, $Q_\alpha$ intervals.
5. Add masses published after AME2020 as a second, truly prospective test set.